### Part 5: The ”News Feed” Extraction Pipeline

### 1. Environment Setup

In [ ]:
import sys
import os
import yaml
import json
import re
import pandas as pd
from pydantic import BaseModel, Field, ValidationError
from typing import Literal, Optional

project_root = os.path.abspath("..")
if project_root not in sys.path:
    sys.path.insert(0, project_root)

from utils.prompts import render
from utils.llm_client import LLMClient

In [ ]:
# Load provider and model tiers from your YAML files
with open("../config/config.yaml", "r") as f:
    config_data = yaml.safe_load(f)

with open("../config/models.yaml", "r") as f:
    models_config = yaml.safe_load(f)

provider = config_data['providers']['default']
# Using the 'reason' tier as specified in the mission goals
client = LLMClient(provider=provider, technique="reason")

### 2. Pydantic Schema Definition

In [ ]:
class CrisisEvent(BaseModel):
    district: Literal["Colombo", "Gampaha", "Kandy", "Kalutara", "Galle", "Matara", "Jaffna", "Kegalle", "Ratnapura", "Nuwara Eliya"] 
    flood_level_meters: Optional[float] = None
    victim_count: int = Field(default=0)
    main_need: str = Field(default="General Relief") # Added default to fix certain items
    status: Literal["Critical", "Warning", "Stable"]

### 3. Process Feed - Utility: JSON Cleaner

In [ ]:
def clean_llm_json(raw_text: str) -> str:
    """Removes markdown blocks and preamble to ensure Pydantic can parse the JSON"""
    clean = re.sub(r"```(?:json)?\s*(.*?)\s*```", r"\1", raw_text, flags=re.DOTALL)
    return clean.strip()

### 4. Extracting Data from News Feed

In [ ]:
input_path = "../data/news_feed.txt"
output_path = "../output/flood_report.xlsx"
os.makedirs("../output", exist_ok=True)

valid_records = []

# Load the 30 mixed items from the feed
if os.path.exists(input_path):
    with open(input_path, "r") as f:
        feed_items = [line.strip() for line in f if line.strip()]
    
    print(f"📡 Processing {len(feed_items)} items")

    for i, item_text in enumerate(feed_items):
        try:
            # Step A: Render prompt and get LLM response
            prompt_text, _ = render("json_extract.v1", text=item_text)
            response = client.chat([{"role": "user", "content": prompt_text}], task_type="classification")
            
            # Step B: Clean the output
            json_str = clean_llm_json(response['text'])
            
            # Step C: Pydantic Validation
            event = CrisisEvent.model_validate_json(json_str)
            
            # Step D: Store valid data
            valid_records.append(event.model_dump())
            print(f"✅ [{i+1}/30] Validated: {event.district}")

        except ValidationError as ve:
            print(f"⚠️ [{i+1}/30] Validation Failed: {ve.errors()[0]['msg']}")
        except Exception as e:
            print(f"❌ [{i+1}/30] Processing Error: {e}")

    # Excel export
    if valid_records:
        df = pd.DataFrame(valid_records)
        df.to_excel(output_path, index=False)
        print(f"\n📊 SUCCESS: {len(valid_records)} items saved to {output_path}")
    else:
        print("\n🚫 No valid data found.")
else:
    print(f"❌ File not found: {input_path}")